# 1. Generate Synthetic User Sessions

**Business goal:** find customers who are about to call customer care, and call them first.

For each user we look at the actions they performed in the **last two hours**, for example:

```
login → check_balance → fx_transfer_attempt → fx_transfer_failed → fx_transfer_attempt → open_help_page
```

Each user gets a label:

- `called = 1` — the user called customer care after this window;
- `called = 0` — the user did not call.

We do not have real data, so in this notebook we **create** it. Because we create it ourselves, we know
the true behavior of each group. Later this lets us check how close the learned model gets to the best possible model.

**Output of this notebook:**

- `data/train.csv` — users for training;
- `data/test.csv` — separate users for evaluation (different random seed);
- `data/true_process.npz` — the true probabilities used to create the data.

## 1.1 Settings

- `CALL_RATE` — the share of users who call. Callers are rare, which is typical for this problem.
- `MIN_ACTIONS`, `MAX_ACTIONS` — how many actions a user performs in the two-hour window.
- `TRAIN_SEED`, `TEST_SEED` — different seeds, so the test users are new users the model has never seen.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

N_TRAIN_USERS = 20_000
N_TEST_USERS = 10_000
CALL_RATE = 0.05                 # 5% of users call customer care
MIN_ACTIONS, MAX_ACTIONS = 3, 20 # number of actions in the two-hour window

TRAIN_SEED = 1
TEST_SEED = 2

DATA_DIR = Path("data")

## 1.2 The actions

| Action | Meaning |
|---|---|
| `login` | The user logs in |
| `check_balance` | Checks the account balance |
| `check_exchange_rates` | Checks currency exchange rates |
| `card_payment` | Pays with a card |
| `transfer_to_own_account` | Moves money between their own accounts |
| `transfer_to_other_user` | Sends money to another user |
| `fx_transfer_attempt` | Tries to make a foreign currency (FX) transfer |
| `fx_transfer_failed` | The FX transfer failed |
| `password_change_attempt` | Tries to change the password |
| `password_change_failed` | The password change failed |
| `open_help_page` | Opens the help page |

In [ ]:
ACTIONS = [
    "login",
    "check_balance",
    "check_exchange_rates",
    "card_payment",
    "transfer_to_own_account",
    "transfer_to_other_user",
    "fx_transfer_attempt",
    "fx_transfer_failed",
    "password_change_attempt",
    "password_change_failed",
    "open_help_page",
]
N_ACTIONS = len(ACTIONS)
ACTION_INDEX = {action: i for i, action in enumerate(ACTIONS)}

## 1.3 How users who do **not** call behave

We describe behavior with **weights**: `"login": {"check_balance": 5, "card_payment": 2}` means that after
`login`, the next action `check_balance` is 2.5 times more likely than `card_payment`.

Later, each row of weights is divided by its sum, so it becomes a probability distribution.
This gives the **transition matrix**: row = current action, column = next action.

Normal users mostly check the balance, pay, and transfer money. Failures happen, but rarely.

In [ ]:
NO_CALL_WEIGHTS = {
    "login": {"check_balance": 5, "check_exchange_rates": 2, "card_payment": 2,
              "transfer_to_own_account": 1, "transfer_to_other_user": 2,
              "fx_transfer_attempt": 1, "password_change_attempt": 0.3},
    "check_balance": {"login": 0.5, "check_balance": 0.5, "check_exchange_rates": 1.5,
                      "card_payment": 2, "transfer_to_own_account": 2,
                      "transfer_to_other_user": 2, "fx_transfer_attempt": 1},
    "check_exchange_rates": {"login": 0.5, "check_balance": 1.5, "check_exchange_rates": 1,
                             "fx_transfer_attempt": 3},
    "card_payment": {"login": 1, "check_balance": 3, "card_payment": 2,
                     "transfer_to_other_user": 0.5},
    "transfer_to_own_account": {"login": 1, "check_balance": 3, "card_payment": 1,
                                "fx_transfer_attempt": 0.5},
    "transfer_to_other_user": {"login": 1, "check_balance": 3, "card_payment": 1,
                               "transfer_to_other_user": 1},
    "fx_transfer_attempt": {"login": 1, "check_balance": 3, "check_exchange_rates": 1,
                            "fx_transfer_failed": 0.4},
    "fx_transfer_failed": {"login": 1, "check_balance": 2, "check_exchange_rates": 1,
                           "fx_transfer_attempt": 2, "open_help_page": 0.3},
    "password_change_attempt": {"login": 3, "check_balance": 1, "password_change_failed": 0.4},
    "password_change_failed": {"login": 1, "password_change_attempt": 2, "open_help_page": 0.3},
    "open_help_page": {"login": 2, "check_balance": 2, "fx_transfer_attempt": 0.5,
                       "password_change_attempt": 0.5},
}

## 1.4 How users who **will** call behave

Users who are about to call behave like normal users, with a few differences. They:

- start FX transfers and password changes more often;
- **fail** much more often;
- **retry** after a failure;
- open the help page after failures, and sometimes stay there.

We write only the transitions that change. All other transitions stay the same as for normal users.

In [ ]:
CALLER_CHANGES = {
    ("login", "fx_transfer_attempt"): 2,
    ("login", "password_change_attempt"): 1,
    ("check_balance", "check_balance"): 1.5,
    ("check_balance", "fx_transfer_attempt"): 2,
    ("fx_transfer_attempt", "fx_transfer_failed"): 2.5,
    ("fx_transfer_failed", "fx_transfer_attempt"): 3,
    ("fx_transfer_failed", "open_help_page"): 1,
    ("password_change_attempt", "password_change_failed"): 2.5,
    ("password_change_failed", "password_change_attempt"): 3,
    ("password_change_failed", "open_help_page"): 1,
    ("open_help_page", "open_help_page"): 1,
}

CALL_WEIGHTS = {action: dict(row) for action, row in NO_CALL_WEIGHTS.items()}
for (from_action, to_action), weight in CALLER_CHANGES.items():
    CALL_WEIGHTS[from_action][to_action] = weight

## 1.5 The first action

The two-hour window can start in the middle of a session, so the first action is not always `login`.

Both groups use the **same** distribution for the first action. So in this data the first action
carries no information about calling. The model will still learn it, and the two estimates should be close.

In [ ]:
FIRST_ACTION_WEIGHTS = {"login": 6, "check_balance": 2, "card_payment": 1,
                        "check_exchange_rates": 1}

## 1.6 From weights to probabilities

Every transition also gets a small `BACKGROUND_WEIGHT`. In real life almost any action can follow any
other action, even if it is rare. It also keeps every true probability above zero.

In [ ]:
BACKGROUND_WEIGHT = 0.05


def weights_to_transition_matrix(weights):
    matrix = np.full((N_ACTIONS, N_ACTIONS), BACKGROUND_WEIGHT)
    for from_action, row in weights.items():
        for to_action, weight in row.items():
            matrix[ACTION_INDEX[from_action], ACTION_INDEX[to_action]] += weight
    return matrix / matrix.sum(axis=1, keepdims=True)


def weights_to_distribution(weights):
    distribution = np.full(N_ACTIONS, BACKGROUND_WEIGHT)
    for action, weight in weights.items():
        distribution[ACTION_INDEX[action]] += weight
    return distribution / distribution.sum()


true_first_action = weights_to_distribution(FIRST_ACTION_WEIGHTS)
true_matrix_no_call = weights_to_transition_matrix(NO_CALL_WEIGHTS)
true_matrix_call = weights_to_transition_matrix(CALL_WEIGHTS)

assert np.allclose(true_matrix_no_call.sum(axis=1), 1)
assert np.allclose(true_matrix_call.sum(axis=1), 1)

pd.DataFrame(true_matrix_call, index=ACTIONS, columns=ACTIONS).round(2)

## 1.7 Look at the two true matrices

The right panel is the **difference** between the groups:

- red = more likely for users who call;
- blue = more likely for users who do not call.

This is the pattern the classifier will try to learn from the data.

In [ ]:
def plot_matrix(ax, matrix, title, cmap="Blues", vmin=0, vmax=1):
    image = ax.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks(range(N_ACTIONS))
    ax.set_xticklabels(ACTIONS, rotation=90)
    ax.set_yticks(range(N_ACTIONS))
    ax.set_yticklabels(ACTIONS)
    ax.set_xlabel("next action")
    ax.set_ylabel("current action")
    return image


difference = true_matrix_call - true_matrix_no_call
limit = np.abs(difference).max()

fig, axes = plt.subplots(1, 3, figsize=(20, 6), constrained_layout=True)
plot_matrix(axes[0], true_matrix_no_call, "True matrix: will NOT call")
image = plot_matrix(axes[1], true_matrix_call, "True matrix: WILL call")
fig.colorbar(image, ax=axes[:2], shrink=0.7)
image = plot_matrix(axes[2], difference, "Difference (call − no call)", cmap="RdBu_r",
                    vmin=-limit, vmax=limit)
fig.colorbar(image, ax=axes[2], shrink=0.7)
plt.show()

## 1.8 Generate one user

1. Draw the first action.
2. Draw each next action from the row of the current action.

The group (call / no call) decides **which matrix** is used.

In [ ]:
def generate_actions(n_actions, transition_matrix, rng):
    actions = [int(rng.choice(N_ACTIONS, p=true_first_action))]
    for _ in range(n_actions - 1):
        current = actions[-1]
        actions.append(int(rng.choice(N_ACTIONS, p=transition_matrix[current])))
    return actions


rng = np.random.default_rng(0)
print("Will NOT call:", [ACTIONS[i] for i in generate_actions(8, true_matrix_no_call, rng)])
print("WILL call:    ", [ACTIONS[i] for i in generate_actions(8, true_matrix_call, rng)])

## 1.9 Generate many users

For each user:

1. Decide if the user will call (probability `CALL_RATE`).
2. Choose how many actions the user performs.
3. Generate the actions with the matrix of the user's group.

**Note:** here the label comes first and then generates the actions. This is exactly what the
classifier assumes, so this first experiment is the "easy case" for the method. Later we can create
data that breaks this assumption and see what happens.

In [ ]:
def generate_users(n_users, seed, id_prefix):
    rng = np.random.default_rng(seed)
    rows = []
    for i in range(n_users):
        called = rng.random() < CALL_RATE
        matrix = true_matrix_call if called else true_matrix_no_call
        n_actions = int(rng.integers(MIN_ACTIONS, MAX_ACTIONS + 1))
        actions = generate_actions(n_actions, matrix, rng)
        rows.append({
            "user_id": f"{id_prefix}_{i}",
            "actions": " ".join(ACTIONS[a] for a in actions),
            "n_actions": n_actions,
            "called": int(called),
        })
    return pd.DataFrame(rows)


train = generate_users(N_TRAIN_USERS, TRAIN_SEED, "train")
test = generate_users(N_TEST_USERS, TEST_SEED, "test")

train.head()

## 1.10 Quick check of the data

- The call rate should be close to `CALL_RATE` in both sets.
- The number of actions has the same distribution in both groups, so length alone does not reveal the label.
- Failures should be more common among callers — but **not only** among callers.
  That is why a simple rule like "any failure means a call" is not enough.

In [ ]:
def summarize(users):
    has_failure = users["actions"].str.contains("failed|open_help_page")
    return users.assign(has_failure_or_help=has_failure).groupby("called").agg(
        users=("user_id", "count"),
        mean_actions=("n_actions", "mean"),
        share_with_failure_or_help=("has_failure_or_help", "mean"),
    ).round(3)


print(f"Train call rate: {train['called'].mean():.3f}")
print(f"Test call rate:  {test['called'].mean():.3f}")
summarize(train)

## 1.11 Save

The true matrices are saved too. The model will **not** use them for training — only for comparison.

In [ ]:
DATA_DIR.mkdir(exist_ok=True)
train.to_csv(DATA_DIR / "train.csv", index=False)
test.to_csv(DATA_DIR / "test.csv", index=False)
np.savez(
    DATA_DIR / "true_process.npz",
    actions=np.array(ACTIONS),
    first_action=true_first_action,
    matrix_no_call=true_matrix_no_call,
    matrix_call=true_matrix_call,
    call_rate=CALL_RATE,
)
print(f"Saved {len(train)} train users and {len(test)} test users to {DATA_DIR}/")